# Demanda mensal do Museu Avila Adobe

## Objetivo

Organizar uma visualização estática para entender em quais meses o Museu Avila Adobe costuma receber mais visitantes.

## Pergunta analisada

Quais meses concentram a maior demanda média de visitantes e podem justificar reforço sazonal de funcionários?

## Descrição dos dados

O conjunto `museum_visitors.csv` contém visitas mensais a museus de Los Angeles. Neste notebook, usei a coluna `Avila Adobe` e a coluna `Date`, que representa o mês de cada observação.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

pd.plotting.register_matplotlib_converters()
%matplotlib inline

sns.set_style("whitegrid")


## Carregamento dos dados

O arquivo é carregado a partir da pasta `data` do repositório, usando caminho relativo ao notebook.


In [ ]:
museum_filepath = Path("../data/museum_visitors.csv")

museum_data = pd.read_csv(museum_filepath, index_col="Date", parse_dates=True)

museum_data.head()


## Visualização original

O gráfico de linha mostra a evolução mensal ao longo do tempo. Ele ajuda a ver a série temporal, mas não responde de forma direta quais meses do ano costumam ter maior demanda.


In [ ]:
plt.figure(figsize=(12,6))

plt.title("Monthly Visitors to Avila Adobe")

# Gráfico de linha mostrando o número de visitantes do Avila Adobe
sns.lineplot(data=museum_data["Avila Adobe"])

plt.xlabel("Date")
plt.ylabel("Avila Adobe")

plt.show()

## Transformação feita

Para responder à pergunta do Lab 1, transformei a data em mês do ano e calculei a média de visitantes do Avila Adobe para cada mês. Também classifiquei os meses em baixa temporada, temporada intermediária e alta temporada, usando limites simples definidos a partir dos valores observados.


In [ ]:
month_order = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
month_map = dict(enumerate(month_order, start=1))

# Dados apenas do Avila Adobe
avila_data = museum_data[["Avila Adobe"]].copy()
avila_data["Mês número"] = avila_data.index.month
avila_data["Mês"] = avila_data["Mês número"].map(month_map)

# Media mensal de visitantes
monthly_visitors = (
    avila_data
    .groupby(["Mês número", "Mês"], as_index=False)["Avila Adobe"]
    .mean()
    .rename(columns={"Avila Adobe": "Média de visitantes"})
)

# Limiares definidos para separar os meses por nivel de demanda
limite_alta_temporada = 26000
limite_temporada_intermediaria = 23000
media_geral = avila_data["Avila Adobe"].mean()

def classificar_mes(valor):
    if valor >= limite_alta_temporada:
        return "Alta temporada"
    elif valor >= limite_temporada_intermediaria:
        return "Temporada intermediária"
    else:
        return "Baixa temporada"

monthly_visitors["Classificação"] = monthly_visitors["Média de visitantes"].apply(classificar_mes)

monthly_visitors

## Justificativa da visualização

Usei barras porque a comparação principal é entre meses. As cores separam os níveis de demanda e a linha tracejada mostra a média geral, ajudando a comparar cada mês com o comportamento médio do período.


In [ ]:
plt.figure(figsize=(12,6))

palette = {
    "Alta temporada": "#D55E00",
    "Temporada intermediária": "#56B4E9",
    "Baixa temporada": "#BDBDBD"
}

# Gráfico de barras mostrando a média de visitantes por mês
ax = sns.barplot(
    data=monthly_visitors,
    x="Mês",
    y="Média de visitantes",
    hue="Classificação",
    hue_order=["Alta temporada", "Temporada intermediária", "Baixa temporada"],
    palette=palette,
    dodge=False
)

# Linha da média geral
plt.axhline(media_geral, color="#6A3D9A", linestyle="--", linewidth=2.2, zorder=1)
plt.text(
    10.35,
    media_geral + 650,
    f"média geral: {media_geral/1000:.1f} mil",
    color="#6A3D9A",
    fontsize=10,
    weight="bold"
)

# Rotulos acima das barras
for container in ax.containers:
    labels = ax.bar_label(
        container,
        fmt=lambda value: f"{value/1000:.1f}k",
        padding=3,
        fontsize=9
    )

    for label in labels:
        label.set_bbox(dict(facecolor="white", edgecolor="none", pad=1.5))
        label.set_zorder(5)

plt.title("Demanda mensal média do Museu Avila Adobe (2014–2018)", fontsize=15, weight="bold", pad=14)
plt.xlabel("Mês")
plt.ylabel("Número de visitantes")

# eixo y
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{value/1000:.0f} mil" if value > 0 else "0"))


plt.legend(title="Classificação", loc="upper left", frameon=True)
sns.despine(left=True, bottom=False)
plt.tight_layout()

# Salva a imagem
plt.savefig("Lab1_ViniciusAlbuquerque_visualizacao.png", dpi=200, bbox_inches="tight")

plt.show()

## Conclusão

Pelo gráfico, maio, julho e agosto aparecem como os meses de maior demanda média do Museu Avila Adobe. Abril, junho, setembro e outubro ficam em uma faixa intermediária. Assim, os dados sugerem que o reforço de equipe faria mais sentido principalmente nos meses de alta temporada, com atenção também para junho, que fica perto desse grupo.
